# Feature Engineering for Hey Banco Clients Dataset

This notebook implements feature engineering for the `hey_clientes.csv` dataset, creating demographic and relationship-based features as specified.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Load the dataset
df = pd.read_csv('../Data/dataset_transacciones/hey_clientes.csv')
print(f"Dataset shape: {df.shape}")
print(df.head())

Dataset shape: (15025, 22)
     user_id  edad sexo            estado                ciudad  \
0  USR-00001    21    M  Ciudad de México  CDMX - Benito Juárez   
1  USR-00002    18    M           Jalisco       Puerto Vallarta   
2  USR-00003    23    H         Chihuahua            Cuauhtémoc   
3  USR-00004    32   SE        Nuevo León             Guadalupe   
4  USR-00005    26    M  Ciudad de México     CDMX - Cuauhtémoc   

  nivel_educativo   ocupacion  ingreso_mensual_mxn  antiguedad_dias  \
0    Preparatoria    Empleado                24500             1554   
1    Preparatoria  Estudiante                19000             1410   
2    Licenciatura  Estudiante                14000             1174   
3        Posgrado    Empleado                61000             1168   
4    Preparatoria  Empresario                27000              816   

   es_hey_pro  ...  score_buro dias_desde_ultimo_login  preferencia_canal  \
0        True  ...         527                       1        app_

## 1. Age Band Feature

Bin `edad` into life stage categories: [18-25, 26-35, 36-50, 51+]

In [4]:
def create_age_band(edad):
    if 18 <= edad <= 25:
        return '18-25'
    elif 26 <= edad <= 35:
        return '26-35'
    elif 36 <= edad <= 50:
        return '36-50'
    elif edad >= 51:
        return '51+'
    else:
        return None

df['age_band'] = df['edad'].apply(create_age_band)
print(df[['edad', 'age_band']].head())

   edad age_band
0    21    18-25
1    18    18-25
2    23    18-25
3    32    26-35
4    26    26-35


## 2. Income Tier Feature

Quantile-bin `ingreso_mensual_mxn` into Q1–Q4 tiers.

In [5]:
df['income_tier'] = pd.qcut(df['ingreso_mensual_mxn'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print(df[['ingreso_mensual_mxn', 'income_tier']].head())

   ingreso_mensual_mxn income_tier
0                24500          Q3
1                19000          Q2
2                14000          Q1
3                61000          Q4
4                27000          Q3


## 3. Engagement Score Feature

Normalize and combine: inverse of `dias_desde_ultimo_login`, `satisfaccion_1_10`, `es_hey_pro`, `nomina_domiciliada`.

In [6]:
# Normalize components
scaler = MinMaxScaler()

# Inverse of days since last login (higher frequency = higher engagement)
df['login_freq'] = 1 / (df['dias_desde_ultimo_login'] + 1)  # +1 to avoid division by zero
df['login_freq_norm'] = scaler.fit_transform(df[['login_freq']])

# Normalize satisfaction (already 1-10, but scale to 0-1)
df['satisfaccion_norm'] = (df['satisfaccion_1_10'] - 1) / 9  # Min 1, max 10

# Binary features to 0-1
df['es_hey_pro_num'] = df['es_hey_pro'].astype(int)
df['nomina_domiciliada_num'] = df['nomina_domiciliada'].astype(int)

# Combine into engagement score (average of normalized components)
df['engagement_score'] = (
    df['login_freq_norm'] + 
    df['satisfaccion_norm'] + 
    df['es_hey_pro_num'] + 
    df['nomina_domiciliada_num']
) / 4

print(df[['dias_desde_ultimo_login', 'satisfaccion_1_10', 'es_hey_pro', 'nomina_domiciliada', 'engagement_score']].head())

   dias_desde_ultimo_login  satisfaccion_1_10  es_hey_pro  nomina_domiciliada  \
0                        1               10.0        True               False   
1                        3                8.0        True               False   
2                        3                8.0        True               False   
3                       16               10.0       False               False   
4                        1                7.0        True               False   

   engagement_score  
0          0.624306  
1          0.505903  
2          0.505903  
3          0.263399  
4          0.540972  


## 4. Credit Health Feature

Segment `score_buro` into categories and normalize the score.

In [7]:
def create_credit_health(score):
    if score < 550:
        return 'poor'
    elif 550 <= score < 650:
        return 'fair'
    elif 650 <= score < 750:
        return 'good'
    elif score >= 750:
        return 'excellent'
    else:
        return None

df['credit_health'] = df['score_buro'].apply(create_credit_health)
# Also normalize the score
df['score_buro_norm'] = scaler.fit_transform(df[['score_buro']])

print(df[['score_buro', 'credit_health', 'score_buro_norm']].head())

   score_buro credit_health  score_buro_norm
0         527          poor         0.418018
1         714          good         0.754955
2         454          poor         0.286486
3         837     excellent         0.976577
4         533          poor         0.428829


## 5. Digital Maturity Feature

Combine `canal_apertura`, `preferencia_canal`, and inverse of `dias_desde_ultimo_login`.

In [8]:
# Map canal_apertura to score (App = 1, Fan Shop = 0)
df['canal_score'] = df['canal_apertura'].map({'App': 1, 'Fan Shop': 0})

# Map preferencia_canal to scores (assuming app_ios/android/huawei are similar)
df['preferencia_score'] = df['preferencia_canal'].map({
    'app_ios': 1, 
    'app_android': 1, 
    'app_huawei': 1
}).fillna(0)  # If other values, assume 0

# Use the same login_freq_norm from engagement score

# Combine into digital maturity score
df['digital_maturity'] = (
    df['canal_score'] + 
    df['preferencia_score'] + 
    df['login_freq_norm']
) / 3

print(df[['canal_apertura', 'preferencia_canal', 'dias_desde_ultimo_login', 'digital_maturity']].head())

  canal_apertura preferencia_canal  dias_desde_ultimo_login  digital_maturity
0            App       app_android                        1          0.832407
1            App       app_android                        3          0.748611
2            App           app_ios                        3          0.748611
3       Fan Shop           app_ios                       16          0.351198
4       Fan Shop           app_ios                        1          0.499074


## 6. Tenure Band Feature

Bin `antiguedad_dias` into loyalty stages: [<90, 90-365, 1-3yr, 3yr+]

In [9]:
def create_tenure_band(dias):
    if dias < 90:
        return '<90'
    elif 90 <= dias < 365:
        return '90-365'
    elif 365 <= dias < 365*3:
        return '1-3yr'
    elif dias >= 365*3:
        return '3yr+'
    else:
        return None

df['tenure_band'] = df['antiguedad_dias'].apply(create_tenure_band)
print(df[['antiguedad_dias', 'tenure_band']].head())

   antiguedad_dias tenure_band
0             1554        3yr+
1             1410        3yr+
2             1174        3yr+
3             1168        3yr+
4              816       1-3yr


## 7. Vulnerability Flag Feature

Flag based on `recibe_remesas`, `ocupacion==Desempleado`, and low income tier (Q1).

In [10]:
df['vulnerability_flag'] = (
    df['recibe_remesas'] | 
    (df['ocupacion'] == 'Desempleado') | 
    (df['income_tier'] == 'Q1')
).astype(int)

print(df[['recibe_remesas', 'ocupacion', 'income_tier', 'vulnerability_flag']].head())

   recibe_remesas   ocupacion income_tier  vulnerability_flag
0           False    Empleado          Q3                   0
1           False  Estudiante          Q2                   0
2           False  Estudiante          Q1                   1
3            True    Empleado          Q4                   1
4           False  Empresario          Q3                   0


## 8. Financial Sophistication Feature

Combine `nivel_educativo`, `usa_hey_shop` (as proxy for investment activity), and `score_buro`.

In [11]:
# Map education level to scores
education_scores = {
    'Secundaria': 1,
    'Preparatoria': 2,
    'Licenciatura': 3,
    'Posgrado': 4
}
df['education_score'] = df['nivel_educativo'].map(education_scores)
df['education_score_norm'] = scaler.fit_transform(df[['education_score']])

# Investment proxy (usa_hey_shop as binary)
df['investment_proxy'] = df['usa_hey_shop'].astype(int)

# Combine with normalized score_buro
df['financial_sophistication'] = (
    df['education_score_norm'] + 
    df['investment_proxy'] + 
    df['score_buro_norm']
) / 3

print(df[['nivel_educativo', 'usa_hey_shop', 'score_buro', 'financial_sophistication']].head())

  nivel_educativo  usa_hey_shop  score_buro  financial_sophistication
0    Preparatoria          True         527                  0.583784
1    Preparatoria          True         714                  0.696096
2    Licenciatura          True         454                  0.651051
3        Posgrado         False         837                  0.658859
4    Preparatoria          True         533                  0.587387


In [13]:
# Display final dataset with all new features
new_features = [
    'age_band', 'income_tier', 'engagement_score', 'credit_health', 
    'digital_maturity', 'tenure_band', 'vulnerability_flag', 'financial_sophistication'
]
print("New engineered features:")
print(df[new_features].head())

# Optional: Save the engineered dataset
df.to_csv('../Data/dataset_transacciones/hey_clientes_engineered.csv', index=False)

New engineered features:
  age_band income_tier  engagement_score credit_health  digital_maturity  \
0    18-25          Q3          0.624306          poor          0.832407   
1    18-25          Q2          0.505903          good          0.748611   
2    18-25          Q1          0.505903          poor          0.748611   
3    26-35          Q4          0.263399     excellent          0.351198   
4    26-35          Q3          0.540972          poor          0.499074   

  tenure_band  vulnerability_flag  financial_sophistication  
0        3yr+                   0                  0.583784  
1        3yr+                   0                  0.696096  
2        3yr+                   1                  0.651051  
3        3yr+                   1                  0.658859  
4       1-3yr                   0                  0.587387  
